<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-PT/blob/main/lingustic_structure_and_interpretability/extract_svo_triples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: Classical Syntax and The Illusion of Comprehension

**Learning Objective:** Understand how classical NLP uses rule-based Dependency Parsing to extract structured information (Subject-Verb-Object) from text.

More importantly, we will empirically expose the fragility of rule-based systems when faced with real-world linguistic variability, setting the stage for the deep learning revolution (Dense Vector Semantics).

### Phase 1: Environment Setup
We use `spaCy`, the industry-standard library for classical NLP pipelines. Run the cell below to install the library and download the small English statistical model.

In [1]:
# Install and download the English model (suppressing output for cleanliness)
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import spacy
from spacy import displacy
from IPython.core.display import display, HTML

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")
print("Environment successfully initialized.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Environment successfully initialized.


### Phase 2: The Illusion of Rule-Based Information Extraction

Before deep learning, we relied on parsing sentences into directed graphs (Dependency Trees) to understand grammatical relationships. Let's build a classic Subject-Verb-Object (SVO) extractor. This was the foundation of early Knowledge Graphs and Question Answering (QA) systems.

We are going to write hardcoded `if/else` rules that look for specific dependency tags (like `subj` for subject and `obj` for object).

In [2]:
def extract_svo_triples(text, nlp_model):
    doc = nlp_model(text)
    svo_triples = []

    for token in doc:
        # Find the Subject
        if "subj" in token.dep_:
            subject = token
            verb = token.head
            # Find the Object attached to the Verb
            for obj in verb.children:
                if "obj" in obj.dep_:
                    svo_triples.append((subject, verb, obj))
                # Handle prepositional objects
                elif obj.dep_ == "prep":
                    for pobj in obj.children:
                        if pobj.dep_ == "pobj":
                            svo_triples.append((subject, verb, pobj))
    return svo_triples

print("SVO Extractor loaded.")

SVO Extractor loaded.


### Phase 3: The Sterile Test

Let's test our rule-based extractor on a sterile, simple dataset. Notice how perfectly it maps the actor to the action.

In [3]:
sterile_text = "The board finalized the acquisition."
sterile_triples = extract_svo_triples(sterile_text, nlp)

print("--- Extracted Triples (Simple Text) ---")
for triple in sterile_triples:
    print(f"Subject: {triple[0].text: <7} | Verb: {triple[1].text: <9} | Object: {triple[2].text}")

--- Extracted Triples (Simple Text) ---
Subject: board   | Verb: finalized | Object: acquisition


### Phase 4: The Stress Test (Real-World Collapse)

The system above looks like artificial intelligence. **It is not.** It is a rigid topological rule.
The real world does not speak in perfect, active SVO structures. Let's see what happens when we introduce standard linguistic variance, specifically the passive voice.

In [4]:
real_world_text = "The aggressive acquisition of the startup was rapidly finalized by the board of directors."
real_world_triples = extract_svo_triples(real_world_text, nlp)

print("--- Extracted Triples (Real-World Text) ---")
if not real_world_triples:
    print("System Failure: No valid SVO triples found based on current rules.")
else:
    for triple in real_world_triples:
        print(f"Parsed -> Subject: {triple[0].text: <11} | Verb: {triple[1].text: <9} | Object: {triple[2].text}")

--- Extracted Triples (Real-World Text) ---
System Failure: No valid SVO triples found based on current rules.


### Phase 5: Visualizing the Algorithmic Blindspot

Why did our extractor fail to identify that the *board* is the entity doing the *finalizing*?
Let's render the dependency tree using `displacy` to look at the exact tags the model assigned to this passive sentence.

**Pay close attention to the tags assigned to "acquisition" and "board".**

In [6]:
# Render the dependency tree
doc_real = nlp(real_world_text)
html = displacy.render(doc_real, style="dep", jupyter=False, options={'distance': 100})
display(HTML(html))

---
### 🛑 YOUR FORUM ASSIGNMENT

In the passive sentence above, the syntax parser labels "acquisition" as the grammatical subject (`nsubjpass`). Our hardcoded logic has completely lost track of who is actually performing the action.

**Your Task:**
1. Open ChatGPT (or any other LLM).
2. Paste the `extract_svo_triples` Python function from Phase 2.
3. Ask the LLM to modify the function by adding the necessary `if/else` rules so that it correctly extracts `(board, finalized, acquisition)` from our `real_world_text`.
4. Run the LLM's code in a new cell below. Did it work?

**To submit in the forum:**
Post the code generated by the LLM. Answer the following critically:
* Exactly how many *new* conditional rules did the LLM have to invent to cover this single edge case?
* Explain fundamentally why relying on syntactic dependency rules will endlessly scale in complexity (an $O(N)$ nightmare) if you were trying to build a Question Answering system for thousands of documents.